In [1]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import h5py
from tensorflow.keras.models import Model
import pandas as pd
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras import backend as keras
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import Recall
from tensorflow.keras.applications.vgg16 import decode_predictions
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.metrics import confusion_matrix,accuracy_score,recall_score
import seaborn as sns
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk

## Interface

In [2]:
def select_folder():
    # 打开文件选择对话框
    
    path = filedialog.askdirectory(title="Select a folder")  # 使用 askdirectory() 可以选择目录
    gallbladder_Identify(path)
    

In [3]:
def gallbladder_Identify(path):
    test_files=[file for file in os.listdir(path) if file.lower().endswith('.jpg')]
    filelist=[]
    test_files_sort=[]
    segment_file=[]
    for filename in test_files:
        filelist.append(filename.split("_")[-1].split(".")[0])
    filedic=dict(zip(filelist,test_files))
    a=sorted(list(map(int,list(filedic.keys()))))
    
    for i in range (1,len(filedic)+1):
        try:
            test_files_sort.append(filedic[str(i)])
        except:
            pass
    test_predict_gen = test_predict_generator(test_files_sort, path,target_size=(224,224))
    
    results = model.predict_generator(test_predict_gen, len(test_files), verbose=1)
    
    threshold = 0.5  # 可以根据需要调整
    
    # 将预测结果解释为类别 
    predicted_classes = [1 if result > threshold else 0 for result in results]
    a=1
    pridict_list=[]
    for i in predicted_classes:
        if i==1:
            pridict_list.append(a)
            print(a)
        a+=1
        
    print("--",len(pridict_list))
    startnum=0
    if len(pridict_list)>1:
        for m in range(0,len(pridict_list)-1):
            if pridict_list[m+1]-pridict_list[m]<=4:
                if startnum==0:
                    first_num=pridict_list[m]
                startnum+=1
                last_num=pridict_list[m+1]
            else:
                break
        for i in range(first_num-2,last_num+3):
            segment_file.append(filedic[str(i)])
    segment_predict(segment_file,path)
    

In [4]:
def segment_predict(segment_file,path):
    global imglist,namelist,masklist,imglist_num,identify_list,identify_num,identify_imgs
    if len(segment_file)>0:
        test_segment1 = test_generator(segment_file,path ,target_size=(512,512))
        result_segment = model_segment1.predict_generator(test_segment1, len(segment_file))
        NewResult=[]
        for item in result_segment:
            img = (item[:, :, 0] * 255.).astype(np.uint8)
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            NewResult.append(img)
        
        imglist_num=0
        identify_num=0
        imglist,namelist,masklist,identify_list=segment_post_processing(NewResult,segment_file,path)
        identify_imgs=size_Adjustment()
        identify_model(identify_imgs)
        image=Image.fromarray(imglist[imglist_num])
        photo = ImageTk.PhotoImage(image)
        segment_label = tk.Label(segment_frame1, image=photo,height=550,width=550)
        segment_label.image = photo
        segment_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
        segment_label2=tk.Label(segment_frame1,text=namelist[imglist_num],font=("Times New Roman",20))
        segment_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
        segment_button=tk.Button(segment_frame1,text="⭠",font=('Times New Roman',20,"bold"),fg="#DCDCDC",relief=tk.FLAT,bg="#808080",width=10,command=button_pre)
        segment_button.grid(row=2, column=2, padx=3,pady=3)
        segment_button2=tk.Button(segment_frame1,text="⭢",font=('Times New Roman',20,"bold"),fg="#DCDCDC",relief=tk.FLAT,bg="#808080",width=10,command=button_next)
        segment_button2.grid(row=2, column=4, padx=3,pady=3)
        
        
        
        
        

In [5]:
def identify_model(identify_imgs):
    global identify_num,identify_results,identify_nums
    identify_predict_gen = identify_predict_generator(identify_imgs,target_size=(224,224))
    identify_results = model_identify1.predict_generator(identify_predict_gen, len(identify_imgs), verbose=1)
    
    threshold = 0.85  # 可以根据需要调整
    # 将预测结果解释为类别
    identify_nums=[]
    identify_predicted_classes = [1 if result > threshold else 0 for result in identify_results]
    for num in range(0,len(identify_predicted_classes)):
        if identify_predicted_classes[num]==1:
            identify_nums.append(num)
    identify_num=0
    
    if len(identify_nums)>0:
        identify_image=Image.fromarray(identify_imgs[identify_nums[identify_num]])
        photo2 = ImageTk.PhotoImage(identify_image)
        identify_label = tk.Label(Identify_frame2, image=photo2,height=250,width=250)
        identify_label.image = photo2
        identify_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
        identify_label2=tk.Label(Identify_frame2,text=str(round(identify_results[identify_nums[identify_num]][0],4)),font=("Times New Roman",20),width=10)
        identify_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
        identify_label3=tk.Label(Identify_frame2,text=namelist[identify_nums[identify_num]],font=("Times New Roman",20))
        identify_label3.grid(row=2, column=2, padx=3,pady=3,columnspan=3)
        identify_label4=tk.Label(Identify_frame2,text="Result:Positive",font=("Times New Roman",20))
        identify_label4.grid(row=4, column=2, padx=3,pady=3,columnspan=3)
    
        identify_button=tk.Button(Identify_frame2,text="⭠",font=('Times New Roman',20,"bold"),fg="#DCDCDC",relief=tk.FLAT,bg="#808080",width=10,command=button_pre2)
        identify_button.grid(row=3, column=2, padx=3,pady=3)
        identify_button2=tk.Button(Identify_frame2,text="⭢",font=('Times New Roman',20,"bold"),fg="#DCDCDC",relief=tk.FLAT,bg="#808080",width=10,command=button_next2)
        identify_button2.grid(row=3, column=4, padx=3,pady=3)
    else:
        identify_image=Image.fromarray(identify_imgs[identify_num])
        photo2 = ImageTk.PhotoImage(identify_image)
        identify_label = tk.Label(Identify_frame2, image=photo2,height=250,width=250)
        identify_label.image = photo2
        identify_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
        identify_label2=tk.Label(Identify_frame2,text=str(round(identify_results[identify_num][0],4)),font=("Times New Roman",20),width=10)
        identify_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
        identify_label3=tk.Label(Identify_frame2,text=namelist[identify_num],font=("Times New Roman",20))
        identify_label3.grid(row=2, column=2, padx=3,pady=3,columnspan=3)
        identify_label4=tk.Label(Identify_frame2,text="Result:Negative",font=("Times New Roman",20))
        identify_label4.grid(row=4, column=2, padx=3,pady=3,columnspan=3)
    
        identify_button=tk.Button(Identify_frame2,text="⭠",font=('Times New Roman',20,"bold"),fg="#DCDCDC",relief=tk.FLAT,bg="#808080",width=10,command=button_pre3)
        identify_button.grid(row=3, column=2, padx=3,pady=3)
        identify_button2=tk.Button(Identify_frame2,text="⭢",font=('Times New Roman',20,"bold"),fg="#DCDCDC",relief=tk.FLAT,bg="#808080",width=10,command=button_next3)
        identify_button2.grid(row=3, column=4, padx=3,pady=3)
    print(identify_results)
    print(identify_predicted_classes)

In [6]:
def size_Adjustment():
    img_list=[]
    for m in range(0,len(identify_list)):
        img=identify_list[m]
        mask=masklist[m]
        index_list=np.argwhere(mask[:,:]>=240)
        cx=0
        cy=0
        for i in index_list:
            cx=cx+i[0]
            cy=cy+i[1]
        cx=round(cx/len(index_list))
        cy=round(cy/len(index_list))
        if cx<=112:
            new_image=img[:224,cy-112:cy+112,:]           
        elif cy<=112:
            print("cy:",cx,cy)
            new_image=img[cx-112:cx+112,:224,:]
        else:
            new_image=img[cx-112:cx+112,cy-112:cy+112,:]
        img_list.append(new_image)

    return img_list

In [7]:
def button_next():
    global imglist_num
    if imglist_num<(len(imglist)-1):
        imglist_num+=1
    image=Image.fromarray(imglist[imglist_num])
    photo = ImageTk.PhotoImage(image)
    segment_label = tk.Label(segment_frame1, image=photo,height=550,width=550)
    segment_label.image = photo
    segment_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
    segment_label2=tk.Label(segment_frame1,text=namelist[imglist_num],font=("Times New Roman",20))
    segment_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)

In [8]:
def button_pre():
    global imglist_num
    if imglist_num>0:
        imglist_num-=1
    image=Image.fromarray(imglist[imglist_num])
    photo = ImageTk.PhotoImage(image)
    segment_label = tk.Label(segment_frame1, image=photo,height=550,width=550)
    segment_label.image = photo
    segment_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
    segment_label2=tk.Label(segment_frame1,text=namelist[imglist_num],font=("Times New Roman",20))
    segment_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)

In [9]:
def button_next2():
    global identify_nums,identify_num
    if identify_num<(len(identify_nums)-1):
        identify_num+=1
    identify_image=Image.fromarray(identify_imgs[identify_nums[identify_num]])
    photo2 = ImageTk.PhotoImage(identify_image)
    identify_label = tk.Label(Identify_frame2, image=photo2,height=250,width=250)
    identify_label.image = photo2
    identify_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
    identify_label2=tk.Label(Identify_frame2,text=str(round(identify_results[identify_nums[identify_num]][0],4)),font=("Times New Roman",20),width=10)
    identify_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
    identify_label3=tk.Label(Identify_frame2,text=namelist[identify_nums[identify_num]],font=("Times New Roman",20))
    identify_label3.grid(row=2, column=2, padx=3,pady=3,columnspan=3)
    


In [10]:
def button_pre2():
    global identify_nums,identify_num
    if identify_num>0:
        identify_num-=1
    identify_image=Image.fromarray(identify_imgs[identify_nums[identify_num]])
    photo2 = ImageTk.PhotoImage(identify_image)
    identify_label = tk.Label(Identify_frame2, image=photo2,height=250,width=250)
    identify_label.image = photo2
    identify_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
    identify_label2=tk.Label(Identify_frame2,text=str(round(identify_results[identify_nums[identify_num]][0],4)),font=("Times New Roman",20),width=10)
    identify_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
    identify_label3=tk.Label(Identify_frame2,text=namelist[identify_nums[identify_num]],font=("Times New Roman",20))
    identify_label3.grid(row=2, column=2, padx=3,pady=3,columnspan=3)

In [11]:
def button_next3():
    global identify_num
    if identify_num<(len(identify_imgs)-1):
        identify_num+=1
    identify_image=Image.fromarray(identify_imgs[identify_num])
    photo2 = ImageTk.PhotoImage(identify_image)
    identify_label = tk.Label(Identify_frame2, image=photo2,height=250,width=250)
    identify_label.image = photo2
    identify_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
    identify_label2=tk.Label(Identify_frame2,text=str(round(identify_results[identify_num][0],4)),font=("Times New Roman",20),width=10)
    identify_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
    identify_label3=tk.Label(Identify_frame2,text=namelist[identify_num],font=("Times New Roman",20))
    identify_label3.grid(row=2, column=2, padx=3,pady=3,columnspan=3)
    

In [12]:
def button_pre3():
    global identify_num
    if identify_num>0:
        identify_num-=1
    identify_image=Image.fromarray(identify_imgs[identify_num])
    photo2 = ImageTk.PhotoImage(identify_image)
    identify_label = tk.Label(Identify_frame2, image=photo2,height=250,width=250)
    identify_label.image = photo2
    identify_label.grid(row=0, column=2, padx=3,pady=3,sticky=tk.E + tk.W + tk.N,columnspan=3)
    identify_label2=tk.Label(Identify_frame2,text=str(round(identify_results[identify_num][0],4)),font=("Times New Roman",20),width=10)
    identify_label2.grid(row=1, column=2, padx=3,pady=3,columnspan=3)
    identify_label3=tk.Label(Identify_frame2,text=namelist[identify_num],font=("Times New Roman",20))
    identify_label3.grid(row=2, column=2, padx=3,pady=3,columnspan=3)

## GB Model Image pre-processing

In [13]:
def test_predict_load_image(test_file, target_size=(224,224)):
    img = cv2.imread(test_file, cv2.IMREAD_COLOR)
    img_size=np.zeros([512,512,3],dtype="uint8")
    img_size[51:461,51:461,:]=img[51:461,51:461,:]
    img = img_size / 255
    img = cv2.resize(img, target_size)
    img = np.reshape(img,(1,) + img.shape)
    return img
def test_predict_generator(test_files, test_data_folder,target_size=(224,224)):
    for test_file in test_files:
        test_file=os.path.join(test_data_folder,test_file)
        yield test_predict_load_image(test_file, target_size)

## Segment Image pre-processing

In [14]:
def test_load_image(test_file, target_size=(512,512)):
    img = cv2.imread(test_file, cv2.IMREAD_GRAYSCALE)
    img_size=np.zeros([512,512],dtype="uint8")
    gaussian_blur = cv2.bilateralFilter(img,15,75,75)
    img_size[51:461,51:461]=gaussian_blur[51:461,51:461]
    img = img_size / 255
    img = cv2.resize(img, target_size)
    img = np.reshape(img, img.shape + (1,))
    img = np.reshape(img,(1,) + img.shape)
    return img

def test_generator(test_files,test_data_folder , target_size=(512,512)):
    for test_file in test_files:
        test_file=os.path.join(test_data_folder,test_file)
        yield test_load_image(test_file, target_size)
        

## AC Model Image pre-processing

In [15]:
def identify_predict_load_image(test_file, target_size=(224,224)):
    img = test_file
    img = img / 255
    img = cv2.resize(img, target_size)
    img = np.reshape(img,(1,) + img.shape)
    return img
def identify_predict_generator(test_files,target_size=(224,224)):
    for test_file in test_files:
        yield identify_predict_load_image(test_file, target_size)

In [16]:
def segment_post_processing(NewResult,segment_file,path):
    kernel = np.ones((15, 15), np.uint8)
    A_file=np.zeros((100,100), np.uint8)
    B_file=np.zeros((100,100), np.uint8)
    num=1
    case_img_list=[]
    case_c_point_list=[]
    file_list_name=[]
    for file_num, item in enumerate(NewResult):
        
        B_file=item
        #B_file_gray = cv2.cvtColor(B_file, cv2.COLOR_BGR2GRAY)
        
        if A_file.shape==B_file.shape:
            A_file_open = cv2.morphologyEx(A_file, cv2.MORPH_OPEN, kernel,iterations=1)
            A_file_open_gray = cv2.cvtColor(A_file_open, cv2.COLOR_BGR2GRAY)
            ret, A_file_open_gray = cv2.threshold(A_file_open_gray, 40, 255, cv2.THRESH_BINARY)
            B_file_open = cv2.morphologyEx(B_file, cv2.MORPH_OPEN, kernel,iterations=1)
            B_file_open_gray = cv2.cvtColor(B_file_open, cv2.COLOR_BGR2GRAY)
            ret, B_file_open_gray = cv2.threshold(B_file_open_gray, 40, 255, cv2.THRESH_BINARY)
            two_img_mask = cv2.bitwise_and(A_file_open_gray, B_file_open_gray)
            contours, hierarchy = cv2.findContours(two_img_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contour_output=A_file_open.copy()
            output=cv2.drawContours(contour_output, contours, -1, (0, 0, 255), 2)
            if len(contours)!= 0:
                contour_num=1
                img_list=[]
                ROI_list=[]
                c_point_list=[]
                for contour in contours:
                    seedpoint_x=0
                    seedpoint_y=0
                    for point in contour:
                        seedpoint_x=seedpoint_x+point[0][0]
                        seedpoint_y=seedpoint_y+point[0][1]      
                    seedpoint=(round(seedpoint_x/len(contour)),round(seedpoint_y/len(contour)))
                    h, w = A_file.shape[:2]
                    ROI = np.zeros((h,w,3), np.uint8)
                    ret, img = cv2.threshold(A_file, 40, 255, cv2.THRESH_BINARY)
                    mask = np.zeros((h+2,w+2,1), np.uint8)
                    cv2.floodFill(img, mask,seedpoint, (0,0,255),(20,20,20),(20,20,20))
                    lower_color = np.array([0, 0, 220])
                    upper_color = np.array([0, 0, 255])
                    ROI = cv2.inRange(img, lower_color, upper_color)
                    if contour_num ==1:
                        index_list=np.argwhere(ROI[:,:]>=240)
                        cx=0
                        cy=0
                        for i in index_list:
                            cx=cx+i[0]
                            cy=cy+i[1]
                        c_point=(round(cy/len(index_list)),round(cx/len(index_list)))
                        if c_point not in c_point_list:
                            img_list.append(img)
                            ROI_list.append(ROI)
                            c_point_list.append(c_point)
                    else:
                        if True in (img != img_list[-1]):
                            index_list=np.argwhere(ROI[:,:]>=240)
                            cx=0
                            cy=0
                            for i in index_list:
                                cx=cx+i[0]
                                cy=cy+i[1]
                            c_point=(round(cy/len(index_list)),round(cx/len(index_list)))
                            if c_point not in c_point_list:
                                img_list.append(img)
                                ROI_list.append(ROI)
                                c_point_list.append(c_point)
                    contour_num=1+contour_num
                case_img_list.append(ROI_list)
                case_c_point_list.append(c_point_list)
                file_list_name.append(segment_file[file_num-1])
            num=num+1
        A_file=B_file
    big_list=[]
    mid_point_num=round(len(case_c_point_list)/2)
    time=0
    
    for point in case_c_point_list[mid_point_num]:
        top_point=True
        bottom_point=True
        small_list=[point]
        for i in range(1,len(case_c_point_list)-mid_point_num):
            if top_point:
                for point_up in case_c_point_list[mid_point_num-i]:
                    if i==1:
                        distance=np.sum(abs(np.array(point_up)-np.array(point)))
                    else:
                        distance=np.sum(abs(np.array(point_up)-np.array(new_point_up)))
                    #print(distance)
                    if distance<70:
                        small_list.append(point_up)
                        new_point_up=point_up
                        top_point=True
                        break
                    else:
                        top_point=False
                        
            if bottom_point:        
                for point_down in case_c_point_list[mid_point_num+i]:
                    if i==1:
                        distance=np.sum(abs(np.array(point_down)-np.array(point)))
                    else:
                        distance=np.sum(abs(np.array(point_down)-np.array(new_point_down)))
                
                    if distance<70:
                        small_list.append(point_down)
                        new_point_down=point_down
                        bottom_point=True
                        break
                    else:
                        bottom_point=False
        big_list.append(small_list)
        
        
    position=[]
    condition=0
    for one_object in big_list:
        if len(one_object)>condition:
            condition=len(one_object)
            gallbladder=one_object
    for a in gallbladder:
        i=0
        for point in case_c_point_list:
            try:
                c=point.index(a)
                position.append([i,c])
            except ValueError:
                pass
            i+=1
    position.sort()
    return_img=[]
    return_mask=[]
    return_filename=[]
    return_identifyimg=[]
    kernel1 = np.ones((7, 7), np.uint8)
    for i in position:
        original = cv2.imread(os.path.join(path,file_list_name[i[0]]))
        ret, mask_out = cv2.threshold(case_img_list[i[0]][i[1]], 200, 255, cv2.THRESH_BINARY)
        edges = cv2.Canny(mask_out, 10, 200)
        Identify_img=np.zeros_like(img)
        mask_out = cv2.erode(mask_out, kernel1)
        Identify_img[mask_out>240]=original[mask_out>240]
        original[edges > 240] = (255,0,0)
        return_identifyimg.append(Identify_img)
        return_img.append(original)
        return_mask.append(mask_out)
        return_filename.append(file_list_name[i[0]])


    return return_img,return_filename,return_mask,return_identifyimg
                
    
    

In [ ]:
model = load_model('gallbladder_VGG16newdata.hdf5', compile=False)
model_segment1 = load_model('unet_gallbladder_seg.hdf5', compile=False)
model_identify1= load_model('gallbladder_VGG16_positive2.hdf5', compile=False)
# 创建主窗口
window=tk.Tk()
window.title("Acute cholecystitis recognition")


# 创建一个顶级菜单
menu_bar = tk.Menu(window)
window.config(menu=menu_bar, relief='groove',height=900,width=1600)

# 创建一个菜单
file_menu = tk.Menu(menu_bar, tearoff=0)
menu_bar.add_cascade(label="File", menu=file_menu)
file_menu.add_command(label="Open...", command=select_folder)
file_menu.add_separator()
file_menu.add_command(label="Exit", command=window.destroy)
segment_frame1 = tk.LabelFrame(window, text="Segmentation", relief=tk.RIDGE,font=("Times New Roman",20),fg="#696969" ,width=800, height=800)
segment_frame1.grid(row=0, column=0, padx=6,pady=6)
Identify_frame2 = tk.LabelFrame(window, text="Identify", relief=tk.RIDGE,font=("Times New Roman",20),fg="#696969" , width=800, height=800)
Identify_frame2.grid(row=0, column=1, padx=6,pady=6)
# 运行主循环
window.mainloop()